# Category 5 - Memory Architecture Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares the memory architecture choices available to Ally Vision v2. The current project uses SQLite plus embedding similarity search because it needs persistent multilingual memory with minimal operational overhead for a local-first assistive application.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
source_urls = {
  "Memory manager": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_manager.py",
  "Memory store": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py",
  "Realtime preload path": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py",
  "Python sqlite3 docs": "https://docs.python.org/3/library/sqlite3.html",
  "Chroma docs": "https://docs.trychroma.com/docs/overview/introduction",
  "Pinecone docs": "https://docs.pinecone.io/docs/overview",
  "Redis docs": "https://redis.io/docs/latest/",
  "LangChain intro": "https://python.langchain.com/docs/introduction/"
}

# Hardcoded public-source values only. No runtime web calls are performed in this notebook.


In [ ]:
comparison_rows = [
  {
    "Metric": "Cross-session persistence",
    "SQLite + embeddings": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
    "In-context only": "No persistent memory service",
    "Vector DB": "Yes [source: https://docs.trychroma.com/docs/overview/introduction]",
    "Redis key-value": "Yes [source: https://redis.io/docs/latest/]",
    "Full RAG": "Yes [source: https://python.langchain.com/docs/introduction/]"
  },
  {
    "Metric": "Extra service required",
    "SQLite + embeddings": "No extra service [source: https://docs.python.org/3/library/sqlite3.html]",
    "In-context only": "No",
    "Vector DB": "Yes",
    "Redis key-value": "Yes",
    "Full RAG": "Usually yes"
  },
  {
    "Metric": "Semantic recall",
    "SQLite + embeddings": "Yes, cosine similarity over embeddings [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
    "In-context only": "Session-only",
    "Vector DB": "Yes",
    "Redis key-value": "Only with extra semantic layer",
    "Full RAG": "Yes"
  },
  {
    "Metric": "Deduplication / update support",
    "SQLite + embeddings": "Yes, similarity and category-aware updates [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
    "In-context only": "No durable layer",
    "Vector DB": "Application-defined",
    "Redis key-value": "Manual overwrite only",
    "Full RAG": "Application-defined"
  },
  {
    "Metric": "Pre-load at session start",
    "SQLite + embeddings": "Yes, priority facts injected before session starts [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
    "In-context only": "No",
    "Vector DB": "Possible",
    "Redis key-value": "Possible",
    "Full RAG": "Possible"
  },
  {
    "Metric": "Monthly infra cost",
    "SQLite + embeddings": "$0 extra service cost [source: https://docs.python.org/3/library/sqlite3.html]",
    "In-context only": "No memory infra but no persistence",
    "Vector DB": "External service cost",
    "Redis key-value": "External service cost",
    "Full RAG": "Highest operational overhead"
  }
]

df = pd.DataFrame(comparison_rows)
display(df)


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
labels=["SQLite+embeddings", "In-context", "Vector DB", "Redis", "Full RAG"]
values=[1, 1, 4, 3, 5]
colors=["#4fc3f7", "#555555", "#555555", "#555555", "#555555"]
fig, ax = plt.subplots(figsize=(10,5))
setup_ax(ax, 'Ally Vision v2 — Memory Setup Complexity Comparison')
ax.bar(labels, values, color=colors)
ax.set_ylabel('Relative setup complexity (lower better)', color='white')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(charts_dir / 'category5_memory_architecture_chart1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
categories=["Persistence", "Startup preload", "Semantic dedupe"]
series={"SQLite+embeddings": {"values": [1, 1, 1], "color": "#4fc3f7"}, "In-context only": {"values": [0, 0, 0], "color": "#555555"}, "Vector DB": {"values": [1, 1, 0.5], "color": "#555555"}, "Redis": {"values": [1, 1, 0], "color": "#555555"}, "Full RAG": {"values": [1, 1, 0.5], "color": "#555555"}}
x=np.arange(len(categories))
width=0.8/max(1,len(series))
fig, ax = plt.subplots(figsize=(12,5))
setup_ax(ax, 'Ally Vision v2 — Persistence / Preload / Dedupe Support')
for idx, (label, spec) in enumerate(series.items()):
    offset=(idx-(len(series)-1)/2)*width
    ax.bar(x+offset, spec['values'], width=width, label=label, color=spec['color'])
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15, ha='right', color='white')
ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white')
plt.tight_layout()
plt.savefig(charts_dir / 'category5_memory_architecture_chart2.png', dpi=150, bbox_inches='tight')
plt.show()


## Data Sources

| # | Source | URL | Accessed via |
|---|--------|-----|-------------|
| 1 | Memory manager | https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_manager.py | project code URL |
| 2 | Memory store | https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py | project code URL |
| 3 | Realtime preload path | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py | project code URL |
| 4 | Python sqlite3 docs | https://docs.python.org/3/library/sqlite3.html | public URL |
| 5 | Chroma docs | https://docs.trychroma.com/docs/overview/introduction | public URL |
| 6 | Pinecone docs | https://docs.pinecone.io/docs/overview | public URL |
| 7 | Redis docs | https://redis.io/docs/latest/ | public URL |
| 8 | LangChain intro | https://python.langchain.com/docs/introduction/ | public URL |


## CONCLUSION

For a local-first assistive technology project, SQLite plus embeddings delivers most of the capability of a hosted vector database with essentially none of the infrastructure burden. The current code already proves the behaviors that matter: persistent facts, semantic recall, preloading, and category-aware updates.

→ Chosen for Ally Vision v2 ✅
